## Wretched creatures in Wollestonecraft and Wollestonecraft Shelley

My initial exploration suggests a commonality in the Wollestonecraft mother and daughter: the appearance of the terms "wretch" and "creature". Let's explore this commonality.

In [1]:
import re
from collections import Counter
import pandas as pd
import spacy
from spacy.matcher import Matcher
nlp = spacy.load('en_core_web_lg')

In [2]:
from IPython.display import display

In [3]:
def clean_gutenberg_text(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    start_pattern = r"\*\*\*\s*START OF.*?\*\*\*"
    end_pattern = r"\*\*\*\s*END OF.*?\*\*\*"

    start_match = re.search(start_pattern, text, re.DOTALL | re.IGNORECASE)
    end_match = re.search(end_pattern, text, re.DOTALL | re.IGNORECASE)

    if not start_match or not end_match:
        raise ValueError("Could not locate Gutenberg boundaries.")

    cleaned = text[start_match.end():end_match.start()]

    return cleaned.strip()

In [4]:
maria_clean = clean_gutenberg_text('../corpuses/wollstonecraft-corpus/maria.txt')

In [5]:
def split_by_contents(text):
    """
    Split a Gutenberg text into sections using its CONTENTS page.

    Returns:
        dict[str, str]
    """

    # Find CONTENTS section
    contents_match = re.search(
        r"\n\s*CONTENTS\s*\n",
        text,
        re.IGNORECASE
    )

    if not contents_match:
        raise ValueError("Could not find CONTENTS section.")

    contents_start = contents_match.end()

    # Everything after CONTENTS
    remainder = text[contents_start:]

    lines = remainder.splitlines()

    titles = []
    first_title_pos = None

    # Collect non-blank lines until we encounter
    # the first title repeated in the body.
    for i, line in enumerate(lines):
        stripped = line.strip()

        if not stripped:
            continue

        titles.append(stripped)

        # Look ahead: if this title appears later surrounded
        # by blank lines, we've probably reached the body.
        pattern = (
            r"\n\s*\n\s*"
            + re.escape(stripped)
            + r"\s*\n"
        )

        m = re.search(pattern, remainder)

        if m and m.start() > 0:
            first_title_pos = m.start()
            break

    if first_title_pos is None:
        raise ValueError("Could not determine end of CONTENTS section.")

    contents_text = remainder[:first_title_pos]
    titles = [
        line.strip()
        for line in contents_text.splitlines()
        if line.strip()
    ]

    body = remainder[first_title_pos:]

    # Locate each title in the body
    sections = []

    for title in titles:
        pattern = (
            r"\n\s*\n\s*"
            + re.escape(title)
            + r"\s*\n"
        )

        m = re.search(pattern, body)

        if m:
            sections.append((title, m.start()))

    if not sections:
        raise ValueError("No section headings found in body.")

    # Slice text into sections
    result = {}

    for i, (title, start) in enumerate(sections):
        if i + 1 < len(sections):
            end = sections[i + 1][1]
        else:
            end = len(body)

        result[title] = body[start:end].strip()

    return result

In [6]:
maria_chapters_dict = split_by_contents(maria_clean)

In [7]:
maria_chapters_dict.keys()

dict_keys(['MARIA', 'PREFACE', 'AUTHOR’S PREFACE', 'CHAPTER 1', 'CHAPTER 2', 'CHAPTER 3', 'CHAPTER 4', 'CHAPTER 5', 'CHAPTER 6', 'CHAPTER 7', 'CHAPTER 8', 'CHAPTER 9', 'CHAPTER 10', 'CHAPTER 11', 'CHAPTER 12', 'CHAPTER 13', 'CHAPTER 14', 'APPENDIX', 'CHAPTER 15', 'CHAPTER 16', 'CHAPTER 17', 'CONCLUSION'])

In [8]:
del(maria_chapters_dict['MARIA'])
del(maria_chapters_dict['PREFACE'])

In [9]:
maria_chapters_dict.keys()

dict_keys(['AUTHOR’S PREFACE', 'CHAPTER 1', 'CHAPTER 2', 'CHAPTER 3', 'CHAPTER 4', 'CHAPTER 5', 'CHAPTER 6', 'CHAPTER 7', 'CHAPTER 8', 'CHAPTER 9', 'CHAPTER 10', 'CHAPTER 11', 'CHAPTER 12', 'CHAPTER 13', 'CHAPTER 14', 'APPENDIX', 'CHAPTER 15', 'CHAPTER 16', 'CHAPTER 17', 'CONCLUSION'])

In [10]:
def unwrap_gutenberg_paragraphs(text):
    """
    Remove Gutenberg line wrapping while preserving paragraphs.
    """

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Split into paragraphs
    paragraphs = re.split(r"\n\s*\n+", text)

    cleaned = []

    for para in paragraphs:
        # Replace internal newlines with spaces
        para = re.sub(r"\s*\n\s*", " ", para)

        # Collapse repeated spaces
        para = re.sub(r"\s+", " ", para)

        cleaned.append(para.strip())

    return "\n\n".join(cleaned)

In [11]:
maria_chapters_dict = {
    title: unwrap_gutenberg_paragraphs(text)
    for title, text in maria_chapters_dict.items()
}

In [12]:
def normalize_whitespace(text):
    """
    Convert all whitespace (spaces, tabs, newlines)
    into single spaces.
    """
    return " ".join(text.split())

In [13]:
maria_chapters_no_pars = {
    title: normalize_whitespace(text)
    for title, text in maria_chapters_dict.items()
}

In [14]:
wretch_pattern = [{"LEMMA": {"IN": ['wretch', 'wretched', 'creature']}}]
matcher = Matcher(nlp.vocab)
matcher.add("WRETCH-CREATURE", [wretch_pattern])

In [15]:
pd.set_option('display.max_colwidth', None)

In [16]:
def extract_emotion_matches(chapters_dict, matcher, nlp):
    match_data = []

    for chapter_number, chapter_text in chapters_dict.items():
        doc = nlp(chapter_text)
        matches = matcher(doc)

        for match_id, start, end in matches:
            span = doc[start:end]
            sentence = span.sent
            label = nlp.vocab.strings[match_id]
            lemma = span[0].lemma_
            cleaned_sentence = " ".join(sentence.text.strip().split())

            match_data.append({
                "chapter": chapter_number,
                "emotion": label,
                "lemma": lemma,
                "sentence": cleaned_sentence,
                "sentence_start_char": sentence.start_char,
                "sentence_end_char": sentence.end_char,
            })

    return pd.DataFrame(match_data)

In [17]:
maria_wretch_df = extract_emotion_matches(maria_chapters_no_pars, matcher, nlp)
maria_wretch_df

,chapter,emotion,lemma,sentence,sentence_start_char,sentence_end_char
0,CHAPTER 1,WRETCH-CREATURE,wretched,"Jemima (she had only a claim to a Christian name, which had not procured her any Christian privileges) could patiently hear of Maria’s confinement on false pretences; she had felt the crushing hand of power, hardened by the exercise of injustice, and ceased to wonder at the perversions of the understanding, which systematize oppression; but, when told that her child, only four months old, had been torn from her, even while she was discharging the tenderest maternal office, the woman awoke in a bosom long estranged from feminine emotions, and Jemima determined to alleviate all in her power, without hazarding the loss of her place, the sufferings of a wretched mother, apparently injured, and certainly unhappy.",10290,11007
1,CHAPTER 2,WRETCH-CREATURE,creature,"An insulated being, from the misfortune of her birth, she despised and preyed on the society by which she had been oppressed, and loved not her fellow-creatures, because she had never been beloved.",2581,2778
2,CHAPTER 2,WRETCH-CREATURE,wretch,"Maria was not permitted to walk in the garden; but sometimes, from her window, she turned her eyes from the gloomy walls, in which she pined life away, on the poor wretches who strayed along the walks, and contemplated the most terrific of ruins—that of a human soul.",4647,4914
3,CHAPTER 2,WRETCH-CREATURE,wretch,"Melancholy and imbecility marked the features of the wretches allowed to breathe at large; for the frantic, those who in a strong imagination had lost a sense of woe, were closely confined.",5979,6168
4,CHAPTER 2,WRETCH-CREATURE,wretch,"Her busy fancy pictured the misery of a fond heart, watching over a friend thus estranged, absent, though present—over a poor wretch lost to reason and the social joys of existence; and losing all consciousness of misery in its excess.",7010,7245
5,CHAPTER 2,WRETCH-CREATURE,wretch,"“They come,” said she, “perhaps, from a wretch condemned, like me, to reason on the nature of madness, by having wrecked minds continually under his eye; and almost to wish himself—as I do—mad, to escape from the contemplation of it.”",9006,9240
6,CHAPTER 2,WRETCH-CREATURE,wretch,"Thus shut out from human intercourse, and compelled to view nothing but the prison of vexed spirits, to meet a wretch in the same situation, was more surely to find a friend, than to imagine a countryman one, in a strange land, where the human voice conveys no information to the eager ear.",11353,11643
7,CHAPTER 2,WRETCH-CREATURE,wretch,"Several minutes elapsed before Maria could enquire respecting the rumour of the house (for this poor wretch was obviously not confined without a cause); and then Jemima could only tell her, that it was said, “she had been married, against her inclination, to a rich old man, extremely jealous (no wonder, for she was a charming creature); and that, in consequence of his treatment, or something which hung on her mind, she had, during her first lying-in, lost her senses.”",14732,15204
8,CHAPTER 2,WRETCH-CREATURE,creature,"Several minutes elapsed before Maria could enquire respecting the rumour of the house (for this poor wretch was obviously not confined without a cause); and then Jemima could only tell her, that it was said, “she had been married, against her inclination, to a rich old man, extremely jealous (no wonder, for she was a charming creature); and that, in consequence of his treatment, or something which hung on her mind, she had, during her first lying-in, lost her senses.”",14732,15204
9,CHAPTER 3,WRETCH-CREATURE,wretch,"Thinking it selfish to dwell on her own sufferings, when in the midst of wretches, who had not only lost all that endears life, but their very selves, her imagination was occupied with melancholy earnestness to trace the mazes of misery, through which so many wretches must have passed to this gloomy receptacle of disjointed souls, to the grand source of human corruption.",

In [18]:
frankenstein_clean = clean_gutenberg_text('../corpuses/shelley-corpus/frankenstein.txt')
frankenstein_chapters = split_by_contents(frankenstein_clean)
frankenstein_normalized = {
    title: normalize_whitespace(text)
    for title, text in frankenstein_chapters.items()
}

In [19]:
frankenstein_wretch_df = extract_emotion_matches(frankenstein_normalized, matcher, nlp)

In [20]:
frankenstein_wretch_df

,chapter,emotion,lemma,sentence,sentence_start_char,sentence_end_char
0,Letter 4,WRETCH-CREATURE,wretched,I never saw a man in so wretched a condition.,3418,3463
1,Letter 4,WRETCH-CREATURE,creature,"I never saw a more interesting creature: his eyes have generally an expression of wildness, and even madness, but there are moments when, if anyone performs an act of kindness towards him or does him any the most trifling service, his whole countenance is lighted up, as it were, with a beam of benevolence and sweetness that I never saw equalled.",4167,4514
2,Letter 4,WRETCH-CREATURE,creature,"He must have been a noble creature in his better days, being even now in wreck so attractive and amiable.",7124,7229
3,Letter 4,WRETCH-CREATURE,creature,How can I see so noble a creature destroyed by misery without feeling the most poignant grief?,7716,7810
4,Letter 4,WRETCH-CREATURE,creature,"“I agree with you,” replied the stranger; “we are unfashioned creatures, but half made up, if one wiser, better, dearer than ourselves—such a friend ought to be—do not lend his aid to perfectionate our weak and faulty natures.",10299,10525
...,...,...,...,...,...,...
112,Chapter 24,WRETCH-CREATURE,creature,"Urged by this view, I refused, and I did right in refusing, to create a companion for the first creature.",32370,32475
113,Chapter 24,WRETCH-CREATURE,wretched,"Miserable himself that he may render no other wretched, he ought to die.",32706,32778
114,Chapter 24,WRETCH-CREATURE,creature,"When I run over the frightful catalogue of my sins, I cannot believe that I am the same creature whose thoughts were once filled with sublime and transcendent visions of the beauty and the majesty of goodness.",40923,41132
115,Chapter 24,WRETCH-CREATURE,wretch,“But it is true that I am a wretch.,42176,42211


In [21]:
for _, row in frankenstein_wretch_df.iterrows():
    print(row['chapter'])
    print(row['lemma'])
    print(row['sentence'])
    print('-------------------')

Letter 4
wretched
I never saw a man in so wretched a condition.
-------------------
Letter 4
creature
I never saw a more interesting creature: his eyes have generally an expression of wildness, and even madness, but there are moments when, if anyone performs an act of kindness towards him or does him any the most trifling service, his whole countenance is lighted up, as it were, with a beam of benevolence and sweetness that I never saw equalled.
-------------------
Letter 4
creature
He must have been a noble creature in his better days, being even now in wreck so attractive and amiable.
-------------------
Letter 4
creature
How can I see so noble a creature destroyed by misery without feeling the most poignant grief?
-------------------
Letter 4
creature
“I agree with you,” replied the stranger; “we are unfashioned creatures, but half made up, if one wiser, better, dearer than ourselves—such a friend ought to be—do not lend his aid to perfectionate our weak and faulty natures.
--------

In [23]:
with pd.ExcelWriter('../outputs/wretched-creatures.xlsx') as writer:
    maria_wretch_df.to_excel(writer, sheet_name='maria')
    frankenstein_wretch_df.to_excel(writer, sheet_name='frankenstein')